# 娇美人揣崽去逼婚，震惊家属院！

**Runtime → Run all**

In [ ]:
!pip install -q requests beautifulsoup4 chardet

In [ ]:
import re, time, requests, chardet
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import files

MAIN_URL = 'http://www.pt9.net/txt209330.html'
OUTPUT   = '娇美人揣崽去逼婚_震惊家属院.txt'
WORKERS  = 8

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/124.0.0.0 Safari/537.36',
    'Accept-Language': 'zh-CN,zh;q=0.9',
    'Referer': 'http://www.pt9.net/',
}
SKIP = re.compile(r'(pt9\.net|请记住本书首发|www\.|本章未完)', re.I)

def smart_decode(resp):
    enc = chardet.detect(resp.content)['encoding'] or 'utf-8'
    try:
        return resp.content.decode(enc)
    except:
        return resp.content.decode('utf-8', errors='replace')

# --- Step 1: inspect main page ---
print('กำลัง fetch หน้าหลัก…')
r = requests.get(MAIN_URL, headers=HEADERS, timeout=15)
html = smart_decode(r)

# Check if it IS already a .txt file
if html.strip().startswith('第') or '第一章' in html[:500]:
    print('พบ plain text โดยตรง! บันทึกเลย…')
    with open(OUTPUT, 'w', encoding='utf-8') as f:
        f.write(html)
    print('✓ เสร็จ!')
    files.download(OUTPUT)
else:
    soup = BeautifulSoup(html, 'html.parser')

    # Try direct txt download link
    txt_link = soup.find('a', href=re.compile(r'\.txt$', re.I))
    if txt_link:
        href = txt_link['href']
        if not href.startswith('http'):
            href = 'http://www.pt9.net' + href
        print(f'พบ .txt download link: {href}')
        r2 = requests.get(href, headers=HEADERS, timeout=60)
        text = smart_decode(r2)
        with open(OUTPUT, 'w', encoding='utf-8') as f:
            f.write(text)
        print('✓ เสร็จ!')
        files.download(OUTPUT)
    else:
        # Find chapter links
        base = 'http://www.pt9.net'
        links = []
        for sel in ['.clist a', '.chapter-list a', '.mulu a', '#list a', 'dl dd a', '.listmain a']:
            links = soup.select(sel)
            if links: break

        if not links:
            # Try all <a> that look like chapter URLs
            links = [a for a in soup.find_all('a', href=True)
                     if re.search(r'/\d+\.html$', a['href'])]

        chapters_urls = []
        for a in links:
            href = a['href']
            if not href.startswith('http'):
                href = base + href
            title = a.get_text(strip=True)
            if title:
                chapters_urls.append((title, href))

        print(f'พบ {len(chapters_urls)} บท')
        if not chapters_urls:
            print('ERROR: ไม่พบ chapter links — แสดง HTML บางส่วนเพื่อ debug:')
            print(html[:2000])
        else:
            def fetch_ch(args):
                i, title, url = args
                try:
                    r = requests.get(url, headers=HEADERS, timeout=15)
                    text = smart_decode(r)
                    s = BeautifulSoup(text, 'html.parser')
                    el = (s.select_one('#content') or s.select_one('#page')
                          or s.select_one('.read-content') or s.select_one('article'))
                    if not el: return i, title, ''
                    for tag in el.select('script,style,.ad,ins'): tag.decompose()
                    lines = [p.get_text(' ', strip=True) for p in el.find_all(['p','br'])]
                    if not lines:
                        lines = [l.strip() for l in el.get_text('\n').splitlines() if l.strip()]
                    return i, title, '\n'.join(l for l in lines if not SKIP.search(l))
                except Exception as e:
                    return i, title, f'[error: {e}]'

            results = {}
            tasks = [(i+1, t, u) for i, (t, u) in enumerate(chapters_urls)]
            with ThreadPoolExecutor(max_workers=WORKERS) as ex:
                futs = {ex.submit(fetch_ch, task): task for task in tasks}
                done = 0
                for f in as_completed(futs):
                    i, title, content = f.result()
                    results[i] = (title, content)
                    done += 1
                    if done % 50 == 0:
                        print(f'  {done}/{len(tasks)} เสร็จ')

            with open(OUTPUT, 'w', encoding='utf-8') as f:
                for i, (title, content) in sorted(results.items()):
                    f.write(f'\n\n第{i}章　{title}\n\n{content}\n')

            print('✓ เสร็จ! กำลังโหลดไฟล์…')
            files.download(OUTPUT)